# TASK-20 · OllamaEmbeddings: query vectors & semantic similarity

Runs independently on a **fresh Colab runtime**. Expect a non-empty vector (768 dimensions for the tested EmbeddingGemma 270M model).

## Setup: hosted accelerator and dependencies

The next cells inspect Colab's optional GPU, install **Python packages**, install required `zstd` for extraction, then install the official **Ollama Linux server**, start its API, and download two small models. On each notebook's fresh runtime the setup repeats. Official installation script executes remote code; only use the official Ollama domain.

In [ ]:
import shutil
import subprocess

print("NVIDIA GPU utility available:", bool(shutil.which("nvidia-smi")))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU assigned — expect CPU inference.")

In [ ]:
%pip install -q 'langchain-ollama>=1.0,<2' 'langchain-core>=1.0,<2' 'ollama>=0.6,<1'

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import urllib.request

def ollama_ready():
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not ollama_ready():
    server_log = open("/tmp/ollama-server.log", "a")
    server = subprocess.Popen(["ollama", "serve"], stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(45):
        if ollama_ready():
            break
        time.sleep(2)
assert ollama_ready(), "Ollama failed to start; inspect /tmp/ollama-server.log"
print("Ollama API responding on localhost:11434")

In [ ]:
import subprocess
for model in ("qwen3:1.7b", "embeddinggemma-2:270m"):
    subprocess.run(["ollama", "pull", model], check=True)
subprocess.run(["ollama", "list"], check=True)

## Query vs document embeddings

`embed_query()` returns one vector and `embed_documents()` returns a list of vectors. `num_ctx` sets input context and `keep_alive=600` keeps the model loaded for 600 seconds.

In [ ]:
from langchain_ollama import OllamaEmbeddings
embedder = OllamaEmbeddings(
    model="embeddinggemma-2:270m",
    base_url="http://localhost:11434",
    num_ctx=4096,
    keep_alive=600,
    validate_model_on_init=True,
)
query = "What is retrieval-augmented generation?"
documents = [
    "RAG retrieves relevant documents to help a language model answer questions.",
    "Neural networks can learn patterns from examples.",
    "Bitcoin is a decentralized digital currency.",
]
query_vector = embedder.embed_query(query)
document_vectors = embedder.embed_documents(documents)
print("Query dimensions:", len(query_vector))
print("Document vector count:", len(document_vectors))
print("First five components:", query_vector[:5])
assert len(query_vector) > 0 and len(document_vectors) == 3

## Cosine similarity

The score measures similarity of vector directions; it cannot verify an answer's factual correctness.

In [ ]:
import math
def cosine_similarity(a, b):
    dot = sum(x*y for x, y in zip(a, b, strict=True))
    na = math.sqrt(sum(x*x for x in a))
    nb = math.sqrt(sum(y*y for y in b))
    return dot/(na*nb) if na and nb else 0.0

for score, document in sorted(
    ((cosine_similarity(query_vector, vec), doc)
     for doc, vec in zip(documents, document_vectors, strict=True)),
    reverse=True
):
    print(f"{score:.4f}  {document}")

## Check processor

Expected: `embeddinggemma-2:270m` loaded on GPU **if Colab assigned a compatible GPU**; otherwise CPU.

In [ ]:
subprocess.run(["ollama", "ps"], check=True)